# Wan 2.2 TI2V-5B Video Generation on AWS Trainium2 (trn2.3xlarge)

End-to-end **text-to-video (T2V)** and **image-to-video (I2V)** generation with **Wan 2.2 TI2V-5B** (5B dense) on a single `trn2.3xlarge` instance at **720P (1280x704)**.

## Environment

| Component | Value |
|-----------|-------|
| **DLAMI** | Deep Learning AMI Neuron (Ubuntu 24.04) 20260410 |
| **Neuron SDK** | 2.29 |
| **Instance** | trn2.3xlarge (LNC=2, 4 logical cores, 96 GB HBM total) |
| **venv** | `/opt/aws_neuronx_venv_pytorch_2_9_nxd_inference/` |

## Architecture

- **Model**: [Wan 2.2 TI2V-5B](https://huggingface.co/Wan-AI/Wan2.2-TI2V-5B-Diffusers) -- 5B dense transformer, supports both T2V and I2V
- **Resolution**: 1280x704 (720P), 81 frames, 50 denoising steps (model trained at 121 frames/24fps)
- **Parallelism**: TP=4 across all 4 cores (world_size=4)
- **Key optimizations**:
  - NKI Flash Attention kernel with SDPA fallback for non-tile-aligned sequences
  - Tiled VAE decoder (384x512 tiles with overlap blending) for 720P
  - Model swapping: load/unload models sequentially to fit 720P in 24 GB HBM per core
  - Dual-timestep I2V: per-position conditioning without OOM (compute t=0 and t=current, blend with mask)
  - `local_rms_norm` workaround for compiler issue with DistributedRMSNorm
  - I2V VAE encoding uses `.mode()` (deterministic argmax, matches official pipeline)

## Pipeline Flow

Because the 720P transformer alone uses ~24 GB HBM per core (nearly all available with LNC=2),
all models cannot be loaded simultaneously. The inference script uses **sequential model swapping**:

```
Phase 0: Load pipeline on CPU (tokenizer, scheduler, VAE for I2V image encoding)
Phase 1: Load text encoder (TP=4) -> encode prompt -> unload text encoder
Phase 2: Load transformer (TP=4) -> 50 denoising steps -> unload transformer
Phase 3: Load PQC + tiled decoder (TP=4) -> decode latents -> save video
```

### I2V: Dual-Timestep Approach

Wan 2.2 TI2V-5B uses `expand_timesteps=True`, meaning per-position timestep conditioning.
For I2V, frame 0 positions need timestep=0 (clean image) while other positions need timestep=t
(current noise level). Naively processing per-position timesteps `[batch, seq_len]` through
the condition embedder creates ~108 MB intermediate tensors per block at 720P, causing OOM.

Our solution: compute condition embeddings for both t=0 and t=current as cheap scalars,
then blend per-position using a mask. This adds only 1.3% NEFF size overhead while providing
correct per-position conditioning.

## Compilation Code

This notebook uses compilation and inference code from [Henan's aws-neuron-samples fork](https://github.com/whn09/aws-neuron-samples/tree/master/torch-neuronx/inference/hf_pretrained_wan2.2_ti2v),
with patches for trn2.3xlarge, I2V dual-timestep support, and SDPA attention fallback.

**Expected time**: ~15 min compilation + ~530s inference (720P, 50 steps)

## Step 1: Environment Setup & Verification

In [ ]:
import os
import subprocess
import sys

# ============================================================
# Configuration -- adjust these for your setup
# ============================================================

# Storage -- trn2.3xlarge has 1x 470 GB NVMe SSD
NVME_MOUNT = "/opt/dlami/nvme"
WORK_DIR = NVME_MOUNT
COMPILED_DIR = f"{WORK_DIR}/compiled_720p"
COMPILER_WORKDIR = f"{WORK_DIR}/compiler_workdir"
CACHE_DIR = f"{WORK_DIR}/wan2.2_ti2v_hf_cache_dir"  # Must match hardcoded path in compile scripts
SAMPLES_DIR = f"{WORK_DIR}/aws-neuron-samples"
HENAN_DIR = f"{SAMPLES_DIR}/torch-neuronx/inference/hf_pretrained_wan2.2_ti2v"
VENV = "/opt/aws_neuronx_venv_pytorch_2_9_nxd_inference"

# Parallelism -- trn2.3xlarge with LNC=2 has 4 logical cores
TP_DEGREE = 4
WORLD_SIZE = 4  # TP only, no CP on 4 cores

# Video generation -- 720P native resolution
HEIGHT = 704
WIDTH = 1280
NUM_FRAMES = 81  # Model trained at 121; 81 is diffusers default, fits trn2.3xlarge memory
NUM_STEPS = 50
FPS = 16  # Model trained at 24fps; 16 gives slow-motion effect at 81 frames

print(f"TP={TP_DEGREE}, World Size={WORLD_SIZE}")
print(f"Resolution: {WIDTH}x{HEIGHT}, {NUM_FRAMES} frames, {NUM_STEPS} steps")
print(f"Work directory: {WORK_DIR}")

In [ ]:
# Verify Neuron devices -- expect 1 device with 4 cores at LNC=2
!neuron-ls

In [ ]:
# Verify SDK versions -- expect SDK 2.29
!pip show neuronx-cc torch-neuronx neuronx-distributed 2>/dev/null | grep -E 'Name:|Version:'

In [ ]:
%%bash -e
# Mount NVMe storage (trn2.3xlarge has 1x 470 GB NVMe SSD)
NVME_MOUNT="/opt/dlami/nvme"

if mountpoint -q "${NVME_MOUNT}" 2>/dev/null; then
    echo "NVMe already mounted at ${NVME_MOUNT}"
else
    NVME_DEVICES=$(lsblk -d -n -o NAME,TYPE | grep disk | grep nvme | awk '{print "/dev/"$1}' | sort)
    ROOT_DEV=$(findmnt -n -o SOURCE / | sed 's/p[0-9]*$//' | sed 's/[0-9]*$//')

    NVME_TO_MOUNT=""
    for dev in $NVME_DEVICES; do
        if [[ "$dev" != "$ROOT_DEV"* ]] && ! lsblk -n -o MOUNTPOINT "$dev" | grep -q '/'; then
            NVME_TO_MOUNT="$dev"
            break
        fi
    done

    if [ -z "$NVME_TO_MOUNT" ]; then
        echo "No unmounted NVMe found, creating directory on EBS instead"
        sudo mkdir -p "${NVME_MOUNT}"
        sudo chown ubuntu:ubuntu "${NVME_MOUNT}"
    else
        echo "Formatting and mounting ${NVME_TO_MOUNT}..."
        sudo mkfs.ext4 -F "$NVME_TO_MOUNT"
        sudo mkdir -p "${NVME_MOUNT}"
        sudo mount "$NVME_TO_MOUNT" "${NVME_MOUNT}"
        sudo chown ubuntu:ubuntu "${NVME_MOUNT}"
        echo "Mounted at ${NVME_MOUNT}"
    fi
fi

df -h ${NVME_MOUNT}

In [ ]:
# Create working directories
for d in [COMPILED_DIR, COMPILER_WORKDIR, CACHE_DIR]:
    os.makedirs(d, exist_ok=True)
print("Directories created.")

## Step 2: Install Dependencies

In [ ]:
# Install required packages
!pip install -q diffusers accelerate imageio imageio-ffmpeg ftfy

In [ ]:
# Patch diffusers: nearest-exact -> nearest (required for Trainium2)
# WanUpsample uses mode="nearest-exact" which triggers _upsample_nearest_exact2d,
# and this fails during XLA tracing on SDK 2.29.
import diffusers

vae_path = os.path.join(
    os.path.dirname(diffusers.__file__),
    "models", "autoencoders", "autoencoder_kl_wan.py"
)
with open(vae_path) as f:
    content = f.read()
if "nearest-exact" in content:
    content = content.replace("nearest-exact", "nearest")
    with open(vae_path, "w") as f:
        f.write(content)
    print("Patched diffusers: nearest-exact -> nearest")
else:
    print("Diffusers already patched")

## Step 3: Clone Neuron Samples & Apply Patches

Henan's code provides the base compilation scripts. We apply patches for:
- **SDPA attention fallback**: non-128-aligned sequence lengths (720P) need `F.scaled_dot_product_attention` instead of NKI flash attention to avoid padding quality loss
- **Dual-timestep I2V**: `timestep_mask` input for per-position conditioning without OOM
- **`run_720p_swap.py`**: Custom inference script for sequential model loading/unloading at 720P
- **`.mode()` VAE encoding**: Correct deterministic encoding for I2V (matches official pipeline)

The patched `compile_transformer.py` and `run_720p_swap.py` are included alongside this notebook.

In [ ]:
%%bash -e
SAMPLES_DIR="/opt/dlami/nvme/aws-neuron-samples"
HENAN_DIR="${SAMPLES_DIR}/torch-neuronx/inference/hf_pretrained_wan2.2_ti2v"

if [ -d "${HENAN_DIR}" ]; then
    echo "aws-neuron-samples already cloned"
else
    echo "Cloning aws-neuron-samples (Henan's fork)..."
    git clone --depth 1 https://github.com/whn09/aws-neuron-samples.git "${SAMPLES_DIR}"
fi

echo "TI2V-5B compilation code:"
ls ${HENAN_DIR}/neuron_wan2_2_ti2v/

echo ""
echo "NOTE: The patched compile_transformer.py and run_720p_swap.py must be copied"
echo "into ${HENAN_DIR}/ before compilation. These files contain:"
echo "  - compile_transformer.py: SDPA fallback + dual-timestep I2V support"
echo "  - run_720p_swap.py: Model-swapping inference for 720P"
echo "Place them alongside this notebook and they will be copied in the next cell."

In [ ]:
%%bash -e
# Copy patched files into the Henan directory
# These files should be placed alongside this notebook before running
HENAN_DIR="/opt/dlami/nvme/aws-neuron-samples/torch-neuronx/inference/hf_pretrained_wan2.2_ti2v"
NOTEBOOK_DIR=$(dirname "$(readlink -f "$0" 2>/dev/null || echo '.')")

# Copy patched compile_transformer.py (SDPA fallback + dual-timestep)
if [ -f compile_transformer.py ]; then
    cp compile_transformer.py ${HENAN_DIR}/neuron_wan2_2_ti2v/compile_transformer.py
    echo "Copied patched compile_transformer.py"
elif [ -f ${HENAN_DIR}/neuron_wan2_2_ti2v/compile_transformer.py ]; then
    echo "Using existing compile_transformer.py (verify it has dual-timestep support)"
fi

# Copy model-swapping inference script
if [ -f run_720p_swap.py ]; then
    cp run_720p_swap.py ${HENAN_DIR}/run_720p_swap.py
    echo "Copied run_720p_swap.py"
elif [ -f ${HENAN_DIR}/run_720p_swap.py ]; then
    echo "Using existing run_720p_swap.py"
fi

echo "\nPatched files in place."

## Step 4: Cache Model (~34 GB)

Downloads the Wan 2.2 TI2V-5B model from HuggingFace into the cache directory
expected by Henan's compilation scripts.

In [ ]:
import torch
from diffusers import AutoencoderKLWan, WanPipeline

if os.path.exists(CACHE_DIR) and len(os.listdir(CACHE_DIR)) > 0:
    print(f"Model already cached at {CACHE_DIR}")
else:
    print(f"Caching model to {CACHE_DIR}...")
    vae = AutoencoderKLWan.from_pretrained(
        "Wan-AI/Wan2.2-TI2V-5B-Diffusers",
        subfolder="vae", torch_dtype=torch.float32, cache_dir=CACHE_DIR)
    pipe = WanPipeline.from_pretrained(
        "Wan-AI/Wan2.2-TI2V-5B-Diffusers",
        vae=vae, torch_dtype=torch.bfloat16, cache_dir=CACHE_DIR)
    del vae, pipe
    print("Cache complete.")

## Step 5: Compile Models for 720P

We compile 4 artifacts for Neuron:
1. **Text encoder** (UMT5-XXL, TP=4) -- ~2-3 min, resolution-independent
2. **Transformer** (5B, TP=4, with dual-timestep I2V support) -- ~5-8 min
3. **Tiled VAE decoder** (384x512 tiles, `--auto-cast=matmult`) -- ~15 min
4. **post_quant_conv** (float32 at full 720P resolution) -- ~1 min

### Important Notes

- **All models must share world_size=4.** NxDModel segfaults if models with different world_sizes are loaded.
- **Do NOT use `neuron_parallel_compile`** -- it overrides world_size to 8.
- **Tiled decoder** is required at 720P because the full-resolution decoder exceeds the per-tile instruction limit.

In [ ]:
# Set up environment for compilation
os.environ["NEURON_RT_VIRTUAL_CORE_SIZE"] = "2"
os.environ["PYTHONPATH"] = HENAN_DIR + ":" + os.environ.get("PYTHONPATH", "")
os.environ["XLA_DISABLE_FUNCTIONALIZATION"] = "1"  # Required for NKI kernels
os.environ["NEURON_FUSE_SOFTMAX"] = "1"
os.environ["NEURON_CUSTOM_SILU"] = "1"

print(f"PYTHONPATH includes: {HENAN_DIR}")
print("Env vars set: XLA_DISABLE_FUNCTIONALIZATION, NEURON_FUSE_SOFTMAX, NEURON_CUSTOM_SILU")

### 5a: Compile Text Encoder

In [ ]:
%%bash -e
HENAN_DIR="/opt/dlami/nvme/aws-neuron-samples/torch-neuronx/inference/hf_pretrained_wan2.2_ti2v"
COMPILED_DIR="/opt/dlami/nvme/compiled_720p"
export PYTHONPATH="${HENAN_DIR}:${PYTHONPATH:-}"
export NEURON_RT_VIRTUAL_CORE_SIZE=2

if [ -f "${COMPILED_DIR}/text_encoder/nxd_model.pt" ]; then
    echo "Text encoder already compiled"
else
    echo "Compiling text encoder (TP=4, world_size=4)..."
    python ${HENAN_DIR}/neuron_wan2_2_ti2v/compile_text_encoder.py \
        --max_sequence_length 512 \
        --tp_degree 4 --world_size 4 \
        --compiled_models_dir ${COMPILED_DIR} \
        2>&1 | tail -10
fi

ls -lh ${COMPILED_DIR}/text_encoder/nxd_model.pt

### 5b: Compile Transformer (720P, dual-timestep for I2V)

The transformer is compiled with a `timestep_mask` input that enables per-position
conditioning for I2V. For T2V, the mask is all-ones (no effect). For I2V, frame 0
positions are 0 (clean image) and all other positions are 1 (noisy).

In [ ]:
%%bash -e
HENAN_DIR="/opt/dlami/nvme/aws-neuron-samples/torch-neuronx/inference/hf_pretrained_wan2.2_ti2v"
COMPILED_DIR="/opt/dlami/nvme/compiled_720p"
COMPILER_WORKDIR="/opt/dlami/nvme/compiler_workdir"
export PYTHONPATH="${HENAN_DIR}:${PYTHONPATH:-}"
export NEURON_RT_VIRTUAL_CORE_SIZE=2
export NEURON_FUSE_SOFTMAX=1
export NEURON_CUSTOM_SILU=1
export XLA_DISABLE_FUNCTIONALIZATION=1

if [ -f "${COMPILED_DIR}/transformer/nxd_model.pt" ]; then
    echo "Transformer already compiled"
else
    echo "Compiling transformer at 720P (1280x704, TP=4, world_size=4)..."
    echo "This will take ~5-8 minutes..."
    python ${HENAN_DIR}/neuron_wan2_2_ti2v/compile_transformer.py \
        --height 704 --width 1280 \
        --num_frames 81 --max_sequence_length 512 \
        --tp_degree 4 --world_size 4 \
        --compiled_models_dir ${COMPILED_DIR} \
        --compiler_workdir ${COMPILER_WORKDIR}/transformer \
        2>&1 | tail -15
fi

ls -lh ${COMPILED_DIR}/transformer/nxd_model.pt

### 5c: Compile Tiled VAE Decoder + post_quant_conv

At 720P, the full-resolution decoder exceeds the Neuron compiler's instruction limit.
We compile the decoder at 384x512 tile size and tile the full 1280x704 latent at runtime
with overlap blending.

The PQC (post_quant_conv) is compiled at full 720P resolution since it is a simple Conv3D.

In [ ]:
%%bash -e
HENAN_DIR="/opt/dlami/nvme/aws-neuron-samples/torch-neuronx/inference/hf_pretrained_wan2.2_ti2v"
COMPILED_DIR="/opt/dlami/nvme/compiled_720p"
COMPILER_WORKDIR="/opt/dlami/nvme/compiler_workdir"
export PYTHONPATH="${HENAN_DIR}:${PYTHONPATH:-}"
export NEURON_RT_VIRTUAL_CORE_SIZE=2
export NEURON_FUSE_SOFTMAX=1
export NEURON_CUSTOM_SILU=1
export XLA_DISABLE_FUNCTIONALIZATION=1

# Patch decoder to use --auto-cast=matmult for better quality
DECODER_SCRIPT="${HENAN_DIR}/neuron_wan2_2_ti2v/compile_decoder_rolling.py"
sed -i 's/--auto-cast=none/--auto-cast=matmult/g' "${DECODER_SCRIPT}"

# Compile tiled decoder (384x512 tiles, non-stateful for tiling)
# Uses --skip_pqc because PQC needs full 720P resolution, not tile size
if [ -f "${COMPILED_DIR}/decoder_tiled/nxd_model.pt" ]; then
    echo "Tiled decoder already compiled"
else
    echo "Compiling tiled VAE decoder (384x512 tiles, auto-cast=matmult)..."
    echo "This will take ~15 minutes..."
    python ${HENAN_DIR}/neuron_wan2_2_ti2v/compile_decoder_rolling.py \
        --height 384 --width 512 \
        --tp_degree 4 --world_size 4 \
        --no_stateful --skip_pqc \
        --compiled_models_dir ${COMPILED_DIR} \
        --compiler_workdir ${COMPILER_WORKDIR}/decoder_tiled \
        --cache_dir /opt/dlami/nvme/wan2.2_ti2v_hf_cache_dir \
        --output_subdir decoder_tiled \
        2>&1 | tail -15
fi

# Compile post_quant_conv at full 720P resolution (uses --skip_decoder)
# PQC is a simple Conv3D that compiles at full resolution without instruction limit issues
if [ -f "${COMPILED_DIR}/post_quant_conv/nxd_model.pt" ]; then
    echo "post_quant_conv already compiled"
else
    echo "Compiling post_quant_conv at 720P..."
    python ${HENAN_DIR}/neuron_wan2_2_ti2v/compile_decoder_rolling.py \
        --height 704 --width 1280 \
        --tp_degree 4 --world_size 4 \
        --skip_decoder \
        --compiled_models_dir ${COMPILED_DIR} \
        --compiler_workdir ${COMPILER_WORKDIR}/pqc \
        --cache_dir /opt/dlami/nvme/wan2.2_ti2v_hf_cache_dir \
        2>&1 | tail -10
fi

echo "\nCompiled artifacts:"
for d in text_encoder transformer decoder_tiled post_quant_conv; do
    if [ -f "${COMPILED_DIR}/${d}/nxd_model.pt" ]; then
        SIZE=$(du -sh "${COMPILED_DIR}/${d}/nxd_model.pt" | cut -f1)
        echo "  ${d}: ${SIZE}"
    else
        echo "  ${d}: MISSING"
    fi
done

## Step 6: Run Inference

We use `run_720p_swap.py` which handles sequential model loading/unloading
to fit 720P within the 24 GB HBM per core limit on trn2.3xlarge.

### Text-to-Video (T2V)

In [ ]:
%%bash -e
HENAN_DIR="/opt/dlami/nvme/aws-neuron-samples/torch-neuronx/inference/hf_pretrained_wan2.2_ti2v"
COMPILED_DIR="/opt/dlami/nvme/compiled_720p"

export PYTHONPATH="${HENAN_DIR}:${PYTHONPATH:-}"
export NEURON_RT_VIRTUAL_CORE_SIZE=2
export NEURON_FUSE_SOFTMAX=1
export NEURON_CUSTOM_SILU=1
export XLA_DISABLE_FUNCTIONALIZATION=1

echo "Running T2V at 720P (1280x704, 81 frames, 50 steps)..."
echo ""

python ${HENAN_DIR}/run_720p_swap.py \
    --compiled_models_dir ${COMPILED_DIR} \
    --height 704 --width 1280 \
    --num_frames 81 \
    --num_inference_steps 50 \
    --prompt "A cat walks on the grass, realistic" \
    --output /opt/dlami/nvme/output_720p_t2v.mp4 \
    --fps 16 \
    --guidance_scale 5.0 \
    2>&1

In [ ]:
from pathlib import Path
from IPython.display import Video, display

video_path = f"{WORK_DIR}/output_720p_t2v.mp4"
if Path(video_path).exists():
    file_size = Path(video_path).stat().st_size / (1024 * 1024)
    print(f"T2V Video: {video_path} ({file_size:.1f} MB)")
    display(Video(video_path, embed=True, width=800))
else:
    print(f"Video not found at {video_path}")

### Image-to-Video (I2V)

Provide an input image and the model will animate it. The VAE encoder for I2V runs on CPU
(Neuron compiler issue NCC_IBIR158 with Conv3D tensorizer). This adds ~5s overhead but
only runs once per video.

The dual-timestep approach gives frame 0 positions timestep=0 conditioning (preserve clean image)
while other positions get timestep=t conditioning (denoise).

**VAE encoding**: Uses `.mode()` (deterministic argmax) matching the official pipeline.
`.sample()` adds random noise which degrades the image condition.

**I2V motion characteristics**: I2V with this 5B model produces subtle, realistic motion
(tail movement, camera drift) rather than dramatic walking. Optical flow analysis shows I2V
actually has *more* true object displacement than T2V (0.66 vs 0.58 mean flow), but less
apparent pixel change (3.72 vs 3.95) because the background is anchored by the input image.
The perception of limited motion is inherent to the model and frame count, not a Neuron issue.

In [ ]:
# Download a sample input image for I2V
import urllib.request
from PIL import Image

input_image_path = f"{WORK_DIR}/input_i2v.png"
if not os.path.exists(input_image_path):
    # Create a sample image from the T2V output (use frame 0)
    # Or download one:
    print("Place your input image at:", input_image_path)
    print("The image will be resized to 1280x704 automatically.")
    print("For best results, use a 16:9 aspect ratio image.")
else:
    img = Image.open(input_image_path)
    print(f"Input image: {img.size}, will be resized to {WIDTH}x{HEIGHT}")

In [ ]:
%%bash -e
HENAN_DIR="/opt/dlami/nvme/aws-neuron-samples/torch-neuronx/inference/hf_pretrained_wan2.2_ti2v"
COMPILED_DIR="/opt/dlami/nvme/compiled_720p"
INPUT_IMAGE="/opt/dlami/nvme/input_i2v.png"

if [ ! -f "${INPUT_IMAGE}" ]; then
    echo "No input image found at ${INPUT_IMAGE}"
    echo "Please provide an input image and re-run this cell."
    exit 0
fi

export PYTHONPATH="${HENAN_DIR}:${PYTHONPATH:-}"
export NEURON_RT_VIRTUAL_CORE_SIZE=2
export NEURON_FUSE_SOFTMAX=1
export NEURON_CUSTOM_SILU=1
export XLA_DISABLE_FUNCTIONALIZATION=1

echo "Running I2V at 720P (1280x704, 81 frames, 50 steps)..."
echo ""

python ${HENAN_DIR}/run_720p_swap.py \
    --compiled_models_dir ${COMPILED_DIR} \
    --height 704 --width 1280 \
    --num_frames 81 \
    --num_inference_steps 50 \
    --image ${INPUT_IMAGE} \
    --prompt "A cat walks on the grass, realistic" \
    --output /opt/dlami/nvme/output_720p_i2v.mp4 \
    --fps 16 \
    --guidance_scale 5.0 \
    2>&1

In [ ]:
video_path = f"{WORK_DIR}/output_720p_i2v.mp4"
if Path(video_path).exists():
    file_size = Path(video_path).stat().st_size / (1024 * 1024)
    print(f"I2V Video: {video_path} ({file_size:.1f} MB)")
    display(Video(video_path, embed=True, width=800))
else:
    print(f"Video not found at {video_path}")

## Performance Summary

### 720P (1280x704) on trn2.3xlarge

| Phase | T2V | I2V |
|-------|-----|-----|
| Text encoding | ~10s | ~10s |
| Image encoding (CPU) | -- | ~5s |
| Transformer (50 steps) | 8.83s/step, 441s | 8.82s/step, 441s |
| Tiled decode | ~77s | ~77s |
| **Total** | **~530s** | **~535s** |

### Key Technical Findings

| Finding | Detail |
|---------|--------|
| NKI attention padding | Non-128-aligned seq_len causes structural quality loss (zero-padded positions dilute softmax). SDPA fallback used for non-aligned sequences. |
| 720P memory constraint | Transformer uses ~24 GB HBM per core. Model swapping (load/unload) frees HBM between phases. |
| I2V per-position timestep | Wan 2.2 5B requires per-position conditioning (`expand_timesteps=True`). Dual-timestep approach avoids OOM while providing correct conditioning. |
| VAE decoder instruction limit | Full-resolution 720P decoder exceeds 1.2M instructions. Tiled decoder (384x512) used instead. |
| `--auto-cast=matmult` | Critical for FP32 models in PyTorch 2.9+. On VAE decoder it prevents precision loss. On transformer, no visible quality difference. |
| Scheduler precision | bf16 vs f32 pipeline state: no visible difference after A/B testing. |
| I2V VAE `.mode()` vs `.sample()` | Official pipeline uses `.mode()` (argmax). Switching from `.sample()` increased motion score from 3.65 to 3.72 (marginal). Always use `.mode()` for correctness. |
| I2V motion level | I2V has more optical flow (0.66) than T2V (0.58) but less pixel diff (3.72 vs 3.95) due to anchored background. Limited walking motion is model-inherent at 81 frames. |
| Guidance scale | 3.0 vs 5.0: no significant motion difference for I2V. Default 5.0 is fine. |
| 121 frames at 720P | **Does not fit on trn2.3xlarge.** TP=4 hits compile-time OOM (29 GB > 24 GB HBM per core). TP=2 CP=2 compiles but text encoder alone uses 23 GB, leaving no room for transformer. 81 frames is the maximum at 720P on this instance. |

### Frame Count and Resolution Limits

The model was trained at 121 frames (24fps, 5s). On trn2.3xlarge with LNC=2:

| Resolution | Max Frames | Reason |
|-----------|-----------|--------|
| 512x512 | 81+ | Fits comfortably |
| 1280x704 (720P) | **81** | 24 GB HBM limit per core |
| 1280x704 + 121f | N/A | Requires trn2.48xlarge or larger |

## Troubleshooting

### Segfault or hang when loading decoder after transformer
**All NxDModels loaded in the same process must share the same world_size.** Recompile
all models with `--world_size 4 --tp_degree 4`.

### `neuron_parallel_compile` overrides world_size
`neuron_parallel_compile` sets its own WORLD_SIZE internally. **Always use `python` directly.**

### "nearest-exact" interpolation error
Run the diffusers patch cell (Step 2) -- this replaces `nearest-exact` with `nearest` in the VAE.

### Out of memory during inference
The model-swapping approach in `run_720p_swap.py` handles this. If you still get OOM:
- Verify LNC=2 is set (`NEURON_LOGICAL_NC_CONFIG=2` in `/etc/environment`)
- Check `neuron-ls` shows 4 cores with 96 GB total
- Ensure no other processes are using the Neuron cores

### Slow first import
On fresh DLAMI instances, the first import of torch-neuronx can take up to 5 minutes
(library rehydration). This is expected.

### I2V green artifacts
If you see green artifacts in I2V output, the transformer was compiled without dual-timestep
support. Recompile with the latest `compile_transformer.py` which includes the `timestep_mask`
input and dual-timestep conditioning logic.

In [ ]:
print("Notebook complete.")
print(f"\nEnvironment summary:")
print(f"  DLAMI: Deep Learning AMI Neuron (Ubuntu 24.04) 20260410")
print(f"  SDK: 2.29")
print(f"  Instance: trn2.3xlarge (LNC=2, 4 cores)")
print(f"  Model: Wan 2.2 TI2V-5B (5B dense)")
print(f"  Resolution: {WIDTH}x{HEIGHT}, {NUM_FRAMES}f, {NUM_STEPS} steps")
print(f"  Parallelism: TP={TP_DEGREE}, WS={WORLD_SIZE}")
print(f"  T2V: 8.83s/step, ~530s total")
print(f"  I2V: 8.82s/step, ~535s total")